# CT-RATE Temporal Labeling v6 — ONE-PASS, SOURCE-AWARE

One MedGemma call per prior→current pair. MedGemma receives both reports, the interval,
the 18 canonical findings, and CT-RATE structured presence transitions. It decides
finding-level clinical meaning semantically—**no keyword/regex gate decides progression**.

Per finding, v6 produces `worsened`, `stable`, `improved`, or `unknown`, plus provenance:
- explicit temporal report statement → real verbatim temporal sentence;
- structured 0→1 / 1→0 fallback → inferred new/resolved + reproducible synthetic sentence;
- silent 1→1 → unknown/no_comparison, never manufactured as stable;
- 0→0 → absent.

**Before running:** A100 80GB; accept MedGemma; upload `ctrate_pairs_enriched_v2.csv`;
run `LIMIT=50` and review QC before setting `LIMIT=0`.

In [ ]:
# 1. Deps
!pip -q install -U "transformers>=4.50" accelerate huggingface_hub

In [ ]:
# 2. GPU sanity — expect an A100 with ~80 GB
import torch
assert torch.cuda.is_available(), 'No GPU! Runtime > Change runtime type > A100 GPU'
p = torch.cuda.get_device_properties(0)
print('GPU:', p.name, f'{p.total_memory/1e9:.0f} GB  torch', torch.__version__)
if p.total_memory/1e9 < 70:
    print('WARNING: <70 GB. bf16 27B needs ~54 GB weights + activations; pick the 80 GB A100.')

In [ ]:
# 3. Hugging Face login (gated MedGemma weights)
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# 4. Upload ctrate_pairs_enriched_v2.csv
from google.colab import files
import csv
csv.field_size_limit(10**9)
up = files.upload()
MANIFEST = list(up.keys())[0]
with open(MANIFEST, newline='', encoding='utf-8') as f:
    ROWS = list(csv.DictReader(f))
print(f'{len(ROWS)} pairs loaded from {MANIFEST}')
need = ['presence_changes', 'prior_labels', 'curr_labels']
missing = [c for c in need if c not in ROWS[0]]
assert not missing, f'Missing {missing}. Upload ctrate_pairs_enriched_v2.csv (run scripts/16 first).'
print('v2 manifest OK.')

In [ ]:
# 5. v6 ONE-PASS prompt + source-aware merge (NO semantic regex)
import json, hashlib
NL = chr(10)
CANON = ['Medical material','Arterial wall calcification','Cardiomegaly',
  'Pericardial effusion','Coronary artery wall calcification','Hiatal hernia',
  'Lymphadenopathy','Emphysema','Atelectasis','Lung nodule','Lung opacity',
  'Pulmonary fibrotic sequela','Pleural effusion','Mosaic attenuation pattern',
  'Peribronchial thickening','Consolidation','Bronchiectasis','Interlobular septal thickening']
REPORT_CHANGES = {'new','worse','stable','improved','resolved','not_temporal'}
DIRECTION = {'new':'worsened','worse':'worsened','stable':'stable',
             'improved':'improved','resolved':'improved'}

SYSTEM_PROMPT = '''You are an expert thoracic radiologist labeling temporal progression in paired CT reports.
For each fixed finding, determine whether the CURRENT report explicitly states how it changed relative to an earlier examination.
Use clinical reasoning and unrestricted radiology language; do not rely on a fixed keyword list. Phrases such as "seen on the old CT but not detected currently," measurements now versus previously, "no longer visualized," and "remains unchanged" may express change.
A static current-study description is NOT an explicit temporal statement. Do not create report evidence by independently comparing static descriptions in the PRIOR and CURRENT reports. Static non-temporal examples include "There is bilateral pleural effusion," "Heart size is normal," "A central venous catheter is present," and "Cardiomegaly is present."
The STRUCTURED TRANSITION is a noisy predicted presence signal. Use it only as fallback when the CURRENT report has no explicit temporal statement. It does not prove severity or stability; present_both NEVER implies stable.
Return exactly one JSON object and no other text.'''

SCHEMA = '''For each finding, first decide whether a CURRENT-report sentence itself explicitly communicates temporal progression.
report_change values:
- new: explicitly newly appeared/developed
- worse: explicitly increased/enlarged/progressed/worsened
- stable: explicitly unchanged/stable/similar/persistent without change/no significant interval change
- improved: explicitly decreased/regressed/partially improved but still present
- resolved: explicitly disappeared/resolved/no longer seen/absent now relative to before
- not_temporal: no explicit temporal statement for this finding

For new/worse/stable/improved/resolved: copy the COMPLETE supporting sentence verbatim from CURRENT REPORT; never paraphrase or invent. Explicit report evidence overrides structured transition.
For not_temporal: leave evidence and temporal_sentence empty. Deterministic code applies the structured fallback and derives the final direction/source fields.
Rules: identical static mentions in prior/current are unknown, not stable. A static current statement with structured new is inferred new but not a temporal sentence. Never infer stable from present_both. Never invent text. Include all 18 findings exactly once and in order.
Return: {"findings":[{"finding":"exact name","report_change":"new|worse|stable|improved|resolved|not_temporal","evidence":"exact short CURRENT quote or empty","temporal_sentence":"complete exact CURRENT sentence or empty"}]}'''

EXAMPLES = '''Examples:
A) PRIOR: No pleural effusion. CURRENT: There is bilateral pleural effusion. STRUCTURED: new. → report_change=not_temporal, empty text. Code later derives inferred new/worsened.
B) CURRENT: Interval increase in the right pleural effusion. → report_change=worse, copy full sentence.
C) PRIOR and CURRENT: Cardiomegaly is present. STRUCTURED: present_both. → report_change=not_temporal, empty text. Code later derives no_comparison/unknown.
D) CURRENT: Bilateral pleural effusion observed in the old CT was not detected in the current examination. → report_change=resolved, copy full sentence.
E) CURRENT: The pulmonary nodule remains unchanged from the previous examination. → report_change=stable, copy full sentence.'''

def curr_text(row): return (row.get('curr_findings','')+' '+row.get('curr_impression','')).strip()
def prior_text(row): return (row.get('prior_findings','')+' '+row.get('prior_impression','')).strip()
def structured_transitions(row):
    try: sparse=json.loads(row.get('presence_changes','') or '{}')
    except Exception: sparse={}
    return {f:sparse.get(f,'absent_both') for f in CANON}
def build_user(row):
    return NL.join([EXAMPLES,'','Now label this pair.','INTERVAL: '+str(row.get('delta_days',''))+' days','',
      'FINDINGS (exact order):','; '.join(CANON),'','STRUCTURED TRANSITIONS (noisy fallback only):',
      json.dumps(structured_transitions(row),ensure_ascii=False),'','PRIOR REPORT:',prior_text(row) or '(none)','',
      'CURRENT REPORT:',curr_text(row) or '(none)','',SCHEMA])
def extract_json(text):
    text=text.replace('```json','').replace('```',''); i=text.find('{')
    if i<0:return None
    depth=0
    for j in range(i,len(text)):
        if text[j]=='{':depth+=1
        elif text[j]=='}':
            depth-=1
            if depth==0:
                try:return json.loads(text[i:j+1])
                except Exception:return None
    return None
def norm_text(s): return ' '.join((s or '').split()).casefold().strip()

# Factual checks only—no keyword/regex decides clinical meaning.
def factual_grounding(row,rc,evidence,sentence):
    if rc not in REPORT_CHANGES:return False,'invalid_report_change'
    if rc=='not_temporal':
        ok=not evidence.strip() and not sentence.strip()
        return ok,('ok' if ok else 'not_temporal_nonempty_text')
    if not sentence.strip():return False,'empty_explicit_sentence'
    current,sent,quote=norm_text(curr_text(row)),norm_text(sentence),norm_text(evidence)
    if sent not in current:return False,'sentence_not_verbatim'
    if quote and quote not in current:return False,'evidence_not_verbatim'
    if quote and quote not in sent:return False,'evidence_not_in_sentence'
    return True,'ok'

SYNTH_SEED=2026
NEW_TEMPLATES=['The {f} is newly present compared with the prior examination.',
 'A new {f} is present on the current examination.',
 'Compared with the prior examination, the {f} is newly present.']
RESOLVED_TEMPLATES=['The {f} has resolved compared with the prior examination.',
 'The previously present {f} is no longer seen.',
 'Compared with the prior examination, the {f} is no longer present.']
def synthetic_sentence(row,finding,change):
    bank=NEW_TEMPLATES if change=='new' else RESOLVED_TEMPLATES
    key=f'{SYNTH_SEED}|{row.get("prior_volume")}|{row.get("curr_volume")}|{finding}|{change}'
    idx=int.from_bytes(hashlib.sha256(key.encode()).digest()[:8],'big')%len(bank)
    return bank[idx].format(f=finding.lower()),idx
def fallback(row,f,p,rc='not_temporal',reason=None,rejected_evidence='',rejected_sentence=''):
    if p=='new': change,direction,source='new','worsened','structured_presence'
    elif p=='resolved': change,direction,source='resolved','improved','structured_presence'
    elif p=='present_both': change,direction,source='no_comparison','unknown','unknown'
    else: change,direction,source='absent','unknown','absent'
    if change in ('new','resolved'):
        sentence,tid=synthetic_sentence(row,f,change); sentence_source='synthetic_presence'
    else: sentence,tid,sentence_source='',None,'none'
    return {'finding':f,'report_change':rc,'change':change,'direction':direction,'label_source':source,
      'presence':p,'evidence':'','report_temporal_sentence':'','temporal_sentence':sentence,
      'temporal_sentence_source':sentence_source,'synthetic_template_id':tid,'report_grounded':False,
      'rejection_reason':reason,'rejected_evidence':rejected_evidence,
      'rejected_temporal_sentence':rejected_sentence}
def combine(row,parsed):
    transitions=structured_transitions(row); proposed={}; duplicates=[]
    if isinstance(parsed,dict):
        for e in parsed.get('findings') or []:
            if not isinstance(e,dict):continue
            f=e.get('finding')
            if f in proposed:duplicates.append(f)
            elif f in CANON:proposed[f]=e
    out=[]
    for f in CANON:
        p=transitions[f]; e=proposed.get(f,{})
        rc=e.get('report_change','not_temporal')
        evidence=(e.get('evidence') or '').strip()[:300]
        sentence=(e.get('temporal_sentence') or '').strip()[:1000]
        grounded,reason=factual_grounding(row,rc,evidence,sentence)
        if grounded and rc in DIRECTION:
            out.append({'finding':f,'report_change':rc,'change':rc,'direction':DIRECTION[rc],
              'label_source':'report_explicit','presence':p,'evidence':evidence,
              'report_temporal_sentence':sentence,'temporal_sentence':sentence,
              'temporal_sentence_source':'real_report','synthetic_template_id':None,
              'report_grounded':True,'rejection_reason':None})
        else:
            reject=reason if (rc!='not_temporal' or not grounded) else None
            out.append(fallback(row,f,p,rc,reject,evidence if reject else '',sentence if reject else ''))
    schema_ok=len(proposed)==len(CANON) and not duplicates and set(proposed)==set(CANON)
    return out,schema_ok,duplicates
print('v6 helpers ready — one semantic pass, no temporal keyword regex')

In [ ]:
# 6. Load MedGemma-27B in bf16 straight onto the GPU
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch, time, gc, os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
for _v in ['model', '_o', '_e', 'out', 'enc']:
    if _v in globals():
        try: del globals()[_v]
        except Exception: pass
gc.collect(); torch.cuda.empty_cache()
free_gb = torch.cuda.mem_get_info()[0] / 1e9
print(f'GPU free before load: {free_gb:.1f} GB')
assert free_gb > 60, ('Only %.1f GB free -> a previous model is still resident. '
                      'Runtime > Restart session, then run cells 1-6 again.' % free_gb)
MODEL_ID = 'google/medgemma-27b-text-it'
tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.padding_side = 'left'
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, attn_implementation='sdpa').to('cuda')
model.config.use_cache = True
model.generation_config.use_cache = True
model.eval()
print('model loaded, bf16, use_cache =', model.config.use_cache)

_p = tok.apply_chat_template([{'role':'user','content':'Reply with the single word: ok'}],
                             tokenize=False, add_generation_prompt=True)
_e = tok(_p, return_tensors='pt').to('cuda')
torch.cuda.synchronize(); _t0 = time.time()
with torch.inference_mode():
    _o = model.generate(**_e, max_new_tokens=64, do_sample=False, use_cache=True,
                        pad_token_id=tok.pad_token_id)
torch.cuda.synchronize(); _dt = time.time() - _t0
_ntok = _o.shape[1] - _e['input_ids'].shape[1]
print(f'WARMUP: {_ntok} tokens in {_dt:.1f}s = {_ntok/max(_dt,1e-6):.1f} tok/s')
print('  -> expect >20 tok/s on the A100.')

In [ ]:
# 7. ONE-PASS labeling — Drive-resumable; reports are NEVER truncated
from google.colab import drive
import os
drive.mount('/content/drive')
DRIVE_DIR='/content/drive/MyDrive/ct_temporal'; os.makedirs(DRIVE_DIR,exist_ok=True)
OUT=os.path.join(DRIVE_DIR,'medgemma_labels_v6.jsonl')
LIMIT=50; BATCH=8; MAX_NEW=2400; RESUME=True
rows_all=ROWS if LIMIT==0 else ROWS[:LIMIT]
done=set()
if RESUME and os.path.exists(OUT):
    for line in open(OUT,encoding='utf-8'):
        try:
            d=json.loads(line); done.add((d['prior_volume'],d['curr_volume']))
        except Exception:pass
rows=[r for r in rows_all if (r['prior_volume'],r['curr_volume']) not in done]
print(f'{len(rows_all)} selected, {len(done)} already done, {len(rows)} to label')
prompts=[tok.apply_chat_template([{'role':'system','content':SYSTEM_PROMPT},{'role':'user','content':build_user(r)}],
          tokenize=False,add_generation_prompt=True) for r in rows]
cfg_context=getattr(model.config,'max_position_embeddings',None); tok_context=getattr(tok,'model_max_length',None)
candidates=[int(x) for x in (cfg_context,tok_context) if isinstance(x,(int,float)) and 0<int(x)<10**7]
MODEL_CONTEXT=min(candidates) if candidates else 8192
if prompts:
    # Match the exact tokenization used below (including any tokenizer-added special tokens).
    prompt_lens=[len(tok(p,add_special_tokens=True)['input_ids']) for p in prompts]; lens=sorted(prompt_lens)
    print(f'prompt tokens median={lens[len(lens)//2]} max={lens[-1]} | context={MODEL_CONTEXT} | reserve={MAX_NEW}')
    too_long=[(i,n) for i,n in enumerate(prompt_lens) if n+MAX_NEW>MODEL_CONTEXT]
    assert not too_long,f'{len(too_long)} prompts exceed context; reports were NOT truncated. First: {too_long[:5]}'
fout=open(OUT,'a' if RESUME and done else 'w',encoding='utf-8'); n_ok=n_bad=n_schema=0
for bi,i in enumerate(range(0,len(prompts),BATCH),1):
    bp,br=prompts[i:i+BATCH],rows[i:i+BATCH]
    enc=tok(bp,return_tensors='pt',padding=True,truncation=False).to('cuda')
    print(f'batch {bi}/{(len(prompts)+BATCH-1)//BATCH} generating...',flush=True)
    torch.cuda.synchronize(); t0=time.time()
    with torch.inference_mode():
        out=model.generate(**enc,max_new_tokens=MAX_NEW,do_sample=False,use_cache=True,pad_token_id=tok.pad_token_id)
    torch.cuda.synchronize(); generated=out[:,enc['input_ids'].shape[1]:]
    generated_width=generated.shape[1]
    texts=tok.batch_decode(generated,skip_special_tokens=True)
    for row,raw in zip(br,texts):
        parsed=extract_json(raw); findings,schema_ok,duplicates=combine(row,parsed); parse_ok=parsed is not None
        n_ok+=int(parse_ok); n_bad+=int(not parse_ok); n_schema+=int(schema_ok)
        rec={'patient':row['patient'],'prior_volume':row['prior_volume'],'curr_volume':row['curr_volume'],
          'delta_days':row['delta_days'],'findings':findings,'parse_ok':parse_ok,'schema_ok':schema_ok,
          'duplicate_findings':duplicates,'label_version':'v6_one_pass',
          'generation_width_tokens':int(generated_width),
          'batch_reached_max_new_tokens':bool(generated_width>=MAX_NEW)}
        if not parse_ok:rec['raw']=raw  # preserve full failed output for cutoff diagnosis
        fout.write(json.dumps(rec,ensure_ascii=False)+NL)
    fout.flush(); print(f'  {time.time()-t0:.1f}s ({i+len(bp)}/{len(prompts)})',flush=True)
fout.close(); print(f'DONE parse_ok={n_ok} fail={n_bad} schema_ok={n_schema} -> {OUT}')

In [ ]:
# 8. v6 QC — source/class/domain/text grounding (NO semantic regex)
from collections import Counter,defaultdict
recs=[json.loads(l) for l in open(OUT,encoding='utf-8') if l.strip()]
print(f'records={len(recs)} parse_ok={sum(r.get("parse_ok",False) for r in recs)} schema_ok={sum(r.get("schema_ok",False) for r in recs)}')
cap_batches=sum(r.get('batch_reached_max_new_tokens',False) for r in recs)
cap_parse_fail=sum(r.get('batch_reached_max_new_tokens',False) and not r.get('parse_ok',False) for r in recs)
print(f'rows from batches reaching MAX_NEW={cap_batches}; parse failures among them={cap_parse_fail}')
keys=['change','direction','label_source','temporal_sentence_source','report_change','rejection_reason']
counts={k:Counter() for k in keys}; by_domain=defaultdict(lambda:{'pairs':0,'direction':Counter(),'source':Counter()})
row_text={(r['prior_volume'],r['curr_volume']):norm_text(curr_text(r)) for r in ROWS}
real_ok=real_bad=0; real_examples=[]; synth_examples=[]; rejected=[]
for rec in recs:
    domain='train' if rec['prior_volume'].startswith('train_') else 'valid' if rec['prior_volume'].startswith('valid_') else 'other'
    by_domain[domain]['pairs']+=1; current=row_text.get((rec['prior_volume'],rec['curr_volume']),'')
    for fd in rec['findings']:
        for k in keys:
            if fd.get(k) is not None:counts[k][fd.get(k)]+=1
        by_domain[domain]['direction'][fd['direction']]+=1; by_domain[domain]['source'][fd['label_source']]+=1
        if fd['temporal_sentence_source']=='real_report':
            ok=bool(norm_text(fd['temporal_sentence'])) and norm_text(fd['temporal_sentence']) in current
            real_ok+=int(ok); real_bad+=int(not ok)
            if len(real_examples)<20:real_examples.append((fd['finding'],fd['change'],fd['temporal_sentence']))
        elif fd['temporal_sentence_source']=='synthetic_presence' and len(synth_examples)<10:
            synth_examples.append((fd['finding'],fd['change'],fd['synthetic_template_id'],fd['temporal_sentence']))
        if fd.get('rejection_reason') and len(rejected)<10:
            rejected.append((fd['finding'],fd['report_change'],fd['rejection_reason'],fd.get('rejected_temporal_sentence','')[:180]))
for k in keys:print(f'{k:26}',dict(counts[k]))
print(f'real-report verbatim: {real_ok}/{real_ok+real_bad} ({100*real_ok/max(real_ok+real_bad,1):.1f}%)')
print('\nBY OFFICIAL DOMAIN')
for domain,d in by_domain.items():print(domain,'pairs',d['pairs'],'direction',dict(d['direction']),'source',dict(d['source']))
print('\nREAL TEMPORAL EXAMPLES — manually inspect semantics'); [print(' ',x) for x in real_examples]
print('\nSYNTHETIC PRESENCE EXAMPLES'); [print(' ',x) for x in synth_examples]
print('\nFACTUAL-GROUNDING REJECTIONS'); [print(' ',x) for x in rejected]
assert all(fd['direction']!='stable' or fd['label_source']=='report_explicit' for r in recs for fd in r['findings'])
assert all(fd['change']!='no_comparison' or fd['direction']=='unknown' for r in recs for fd in r['findings'])
assert all(fd['temporal_sentence_source']!='real_report' or fd['report_grounded'] for r in recs for fd in r['findings'])
assert cap_parse_fail==0, ('Possible truncated generation detected. Inspect full raw output; '
                          'increase MAX_NEW only if the context-window check permits it.')
print('\nINVARIANTS PASSED: stable explicit only; no_comparison unknown; real text grounded.')

In [ ]:
# 9. Output is already on Drive; download a backup
from google.colab import files
print(OUT, 'lines:', sum(1 for _ in open(OUT)))
files.download(OUT)

## v6 pilot QC checklist

Before `LIMIT=0`: parse/schema should be high; real-report verbatim should be 100%; manually
inspect all printed real temporal examples; static statements must be `not_temporal`; “old CT …
not detected” must survive; every stable label must be report-explicit; silent present-both must
be unknown; structured new/resolved must have fixed synthetic text and template IDs; review
train/valid domain counts. Output: `MyDrive/ct_temporal/medgemma_labels_v6.jsonl`.